# Improving Recognition of Rare Postural Transitions

CS3244 HAPT experiment notebook · 23 October 2026

## Objective and summary

The six postural transitions are rare relative to the six steady activities. This notebook consolidates the work in this attempt: participant-grouped evaluation; one-window, causal two-window, and previous/current/following context; mild class weighting; random oversampling; five model families; and a PCA comparison.

The primary metric is macro-F1 over activity IDs 7–12 (transition macro-F1). Transition macro-recall is secondary; common-class macro-F1 and accuracy provide context. The most consistent gain came from three-window temporal context. Mild weighting and random oversampling had smaller, model-dependent effects. The best-supported model for transition recognition was logistic regression with three-window context and mild weighting.

## Data and evaluation design

The notebook uses Minh’s engineered HAPT feature CSVs stored in `wrangled data/`. Each row contains participant ID, activity ID/name, and 561 engineered features. IDs are used for participant-grouped splitting and to prevent context windows crossing participant boundaries; they are not predictor features.

Validation uses `StratifiedGroupKFold` on the 21 training participants. The primary ablation uses three folds with seed 42; selected checks repeat three-fold validation with seeds 123 and 2026. Scaling and optional PCA are fitted on each training fold only. Random oversampling is applied only after the train/validation split. The nine-participant test split was examined earlier in the project, so test results here are descriptive rather than a pristine final evaluation.

The previous/current/following representation is offline: it requires the following window. Rows are assumed to be in temporal order within each participant. The supplied files do not mark every within-participant experiment boundary, so some adjacent rows may cross a protocol boundary.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

candidates = [Path.cwd() / "wrangled data", Path.cwd() / "23_10_26 attempt_yifei" / "wrangled data"]
DATA_DIR = next((p for p in candidates if (p / "hapt_train.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find wrangled data/hapt_train.csv")

train_df = pd.read_csv(DATA_DIR / "hapt_train.csv")
test_df = pd.read_csv(DATA_DIR / "hapt_test.csv")
META_COLUMNS = ["subject_id", "activity_id", "activity_label"]
FEATURE_COLUMNS = [c for c in train_df.columns if c not in META_COLUMNS]
LABELS = np.arange(1, 13)
TRANSITION_LABELS = np.arange(7, 13)

assert len(FEATURE_COLUMNS) == 561
assert set(test_df.columns) == set(train_df.columns)
print(f"Train: {train_df.shape}; test: {test_df.shape}; features: {len(FEATURE_COLUMNS)}")
print(f"Train participants: {train_df.subject_id.nunique()}; test participants: {test_df.subject_id.nunique()}")
print(f"Participant overlap: {set(train_df.subject_id) & set(test_df.subject_id)}")
train_class_counts = train_df.groupby(["activity_id", "activity_label"]).size().rename("windows")
test_class_counts = test_df.groupby(["activity_id", "activity_label"]).size().rename("windows")
display(pd.concat({"train": train_class_counts, "test": test_class_counts}, axis=1).fillna(0).astype(int))

Train: (7767, 564); test: (3162, 564); features: 561
Train participants: 21; test participants: 9
Participant overlap: set()


,,train,test
activity_id,activity_label,,
1,WALKING,1226,496
2,WALKING_UPSTAIRS,1073,471
3,WALKING_DOWNSTAIRS,987,420
4,SITTING,1293,508
5,STANDING,1423,556
6,LAYING,1413,545
7,STAND_TO_SIT,47,23
8,SIT_TO_STAND,23,10
9,SIT_TO_LIE,75,32


## Feature construction and metrics

For each participant, a context row is formed by concatenating neighboring feature vectors. At participant boundaries the missing neighbor is replaced by the current row. Mild class weights are proportional to the training-fold class frequency raised to −0.5 and normalized to mean 1. Oversampling duplicates transition examples with replacement until each selected transition class reaches a specified row target.

In [2]:
def arrays(frame):
    x = frame[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    y = frame.activity_id.to_numpy(dtype=np.int64)
    groups = frame.subject_id.to_numpy(dtype=np.int64)
    return x, y, groups


def add_context(x, groups, mode="single"):
    if mode == "single":
        return x
    previous = np.vstack((x[:1], x[:-1])).copy()
    previous[np.flatnonzero(groups[1:] != groups[:-1]) + 1] = x[np.flatnonzero(groups[1:] != groups[:-1]) + 1]
    if mode == "causal":
        return np.hstack((x, previous))
    if mode != "bidirectional":
        raise ValueError(f"Unknown context mode: {mode}")
    following = np.vstack((x[1:], x[-1:])).copy()
    boundary = np.flatnonzero(groups[:-1] != groups[1:])
    following[boundary] = x[boundary]
    return np.hstack((previous, x, following))


def class_weights(y, exponent=0.5):
    counts = np.bincount(y, minlength=13)
    sample_weight = counts[y].astype(float) ** (-exponent)
    return sample_weight / sample_weight.mean()


def oversample_indices(y, target, seed=42):
    rng = np.random.default_rng(seed)
    indices = [np.arange(len(y))]
    for label in TRANSITION_LABELS:
        found = np.flatnonzero(y == label)
        if len(found) < target:
            indices.append(rng.choice(found, size=target - len(found), replace=True))
    return np.concatenate(indices)


def scores(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    return {
        "transition_f1": float(f1[6:].mean()),
        "transition_recall": float(recall[6:].mean()),
        "common_f1": float(f1[:6].mean()),
        "macro_f1": float(f1.mean()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "per_class_f1": f1,
        "per_class_recall": recall,
        "per_class_precision": precision,
        "support": support,
    }

x_train, y_train, groups_train = arrays(train_df)
x_test, y_test, groups_test = arrays(test_df)
assert set(groups_train).isdisjoint(groups_test)
print("Context feature counts:", {m: add_context(x_train[:3], np.repeat(1, 3), m).shape[1] for m in ["single", "causal", "bidirectional"]})

Context feature counts: {'single': 561, 'causal': 1122, 'bidirectional': 1683}


## Models and reproducible evaluation functions

Fixed configurations used in the matched comparisons: logistic regression (`C=0.1`); linear SVM (`C=0.01`); random forest (200 trees); XGBoost (300 trees, depth 6, learning rate 0.138, subsample 0.616, column sample 0.847, gamma 0.00075, minimum child weight 7); and a 256–128–64 MLP with batch normalization, dropout, weighted cross-entropy, Adam at 0.001, and 20 epochs for the main comparison. MLP epoch and PCA sensitivity were also checked separately.

The functions below reproduce the shared five-model protocol. XGBoost and MLP dependencies are imported only when those models are run. The logged results shown later are the completed experiment outputs; rerunning all folds is optional and can take substantial time.

In [3]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


def prepare(x_train, x_eval, representation="raw"):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(x_train)
    eval_scaled = scaler.transform(x_eval)
    if representation == "pca200":
        pca = PCA(n_components=200, svd_solver="randomized", random_state=42)
        train_scaled = pca.fit_transform(train_scaled)
        eval_scaled = pca.transform(eval_scaled)
    return train_scaled.astype(np.float32), eval_scaled.astype(np.float32)


def mlp_predict(x_train, y_train, sample_weight, x_eval, epochs=20):
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    torch.set_num_threads(2)
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Linear(x_train.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 12),
    )
    for layer in model:
        if isinstance(layer, nn.Linear):
            if layer.out_features == 12:
                nn.init.xavier_uniform_(layer.weight)
            else:
                nn.init.kaiming_normal_(layer.weight)
            nn.init.zeros_(layer.bias)
    if sample_weight is None:
        sample_weight = np.ones(len(y_train), dtype=np.float32)
    data = TensorDataset(
        torch.from_numpy(x_train),
        torch.from_numpy((y_train - 1).astype(np.int64)),
        torch.from_numpy(sample_weight.astype(np.float32)),
    )
    loader = DataLoader(data, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(42))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(reduction="none")
    model.train()
    for _ in range(epochs):
        for batch_x, batch_y, batch_weight in loader:
            optimizer.zero_grad()
            loss = (loss_fn(model(batch_x), batch_y) * batch_weight).mean()
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        return model(torch.from_numpy(x_eval)).argmax(dim=1).numpy() + 1


def predict(model_name, x_train, y_train, x_eval, sample_weight=None, epochs=20):
    if model_name == "mlp":
        return mlp_predict(x_train, y_train, sample_weight, x_eval, epochs)
    if model_name == "logreg":
        model = LogisticRegression(C=0.1, solver="lbfgs", max_iter=600)
    elif model_name == "linear_svm":
        model = LinearSVC(C=0.01, dual=False, max_iter=2000)
    elif model_name == "random_forest":
        model = RandomForestClassifier(n_estimators=200, n_jobs=2, random_state=42)
    elif model_name == "xgboost":
        from xgboost import XGBClassifier
        model = XGBClassifier(
            objective="multi:softprob", num_class=12, tree_method="hist", n_jobs=2,
            n_estimators=300, max_depth=6, learning_rate=0.138,
            subsample=0.616, colsample_bytree=0.847, gamma=0.00075,
            min_child_weight=7, random_state=42,
        )
        model.fit(x_train, y_train - 1, sample_weight=sample_weight)
        return model.predict(x_eval) + 1
    else:
        raise ValueError(model_name)
    model.fit(x_train, y_train, sample_weight=sample_weight)
    return model.predict(x_eval)


def evaluate_cv(model_name="logreg", context="bidirectional", weight_exponent=0.5,
                oversample_target=0, representation="raw", epochs=20, seed=42):
    features = add_context(x_train, groups_train, context)
    splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed)
    rows = []
    for fold, (tr, va) in enumerate(splitter.split(features, y_train, groups_train), 1):
        assert set(groups_train[tr]).isdisjoint(groups_train[va])
        fold_x, fold_y = features[tr], y_train[tr]
        sample_weight = class_weights(fold_y, weight_exponent) if weight_exponent else None
        if oversample_target:
            idx = oversample_indices(fold_y, oversample_target, seed=42 + fold)
            fold_x, fold_y = fold_x[idx], fold_y[idx]
            if sample_weight is not None:
                sample_weight = sample_weight[idx]
        fit_x, val_x = prepare(fold_x, features[va], representation)
        pred = predict(model_name, fit_x, fold_y, val_x, sample_weight, epochs)
        rows.append({"fold": fold, **scores(y_train[va], pred)})
    return pd.DataFrame(rows)


def evaluate_test(model_name="logreg", context="bidirectional", weight_exponent=0.5,
                  oversample_target=0, representation="raw", epochs=20):
    train_features = add_context(x_train, groups_train, context)
    test_features = add_context(x_test, groups_test, context)
    fit_y = y_train.copy()
    sample_weight = class_weights(fit_y, weight_exponent) if weight_exponent else None
    if oversample_target:
        idx = oversample_indices(fit_y, oversample_target, seed=42)
        train_features, fit_y = train_features[idx], fit_y[idx]
        if sample_weight is not None:
            sample_weight = sample_weight[idx]
    fit_x, eval_x = prepare(train_features, test_features, representation)
    pred = predict(model_name, fit_x, fit_y, eval_x, sample_weight, epochs)
    return scores(y_test, pred)

# Example reruns (not executed automatically):
# evaluate_cv("logreg", context="bidirectional", weight_exponent=0.5)
# evaluate_test("random_forest", context="bidirectional", weight_exponent=0.5, oversample_target=60)

## Results: controlled ablation

All values below are transition macro-F1. Baseline uses one window without class weighting; the other columns isolate mild weighting and three-window context. The final column combines context and weighting. Validation values are the seed-42 three-fold participant-grouped means.

In [4]:
models = ["Logistic regression", "Linear SVM", "Random forest", "XGBoost", "MLP (20 epochs)"]
validation_ablation = pd.DataFrame({
    "Model": models,
    "Baseline": [0.7433, 0.7367, 0.6411, 0.6774, 0.7343],
    "Weights only": [0.7385, 0.7451, 0.6787, 0.7561, 0.7419],
    "Three windows only": [0.8540, 0.8503, 0.8131, 0.7954, 0.8310],
    "Three windows + weights": [0.8631, 0.8649, 0.8236, 0.8398, 0.8617],
})
validation_ablation

,Model,Baseline,Weights only,Three windows only,Three windows + weights
0,Logistic regression,0.7433,0.7385,0.8540,0.8631
1,Linear SVM,0.7367,0.7451,0.8503,0.8649
2,Random forest,0.6411,0.6787,0.8131,0.8236
3,XGBoost,0.6774,0.7561,0.7954,0.8398
4,MLP (20 epochs),0.7343,0.7419,0.8310,0.8617


In [5]:
test_ablation = pd.DataFrame({
    "Model": models,
    "Baseline": [0.7532, 0.8112, 0.7120, 0.7888, 0.7291],
    "Weights only": [0.7249, 0.7807, 0.7532, 0.7851, 0.8139],
    "Three windows only": [0.9045, 0.8818, 0.8717, 0.8853, 0.8626],
    "Three windows + weights": [0.8966, 0.8778, 0.8727, 0.8463, 0.8632],
})
test_ablation

,Model,Baseline,Weights only,Three windows only,Three windows + weights
0,Logistic regression,0.7532,0.7249,0.9045,0.8966
1,Linear SVM,0.8112,0.7807,0.8818,0.8778
2,Random forest,0.7120,0.7532,0.8717,0.8727
3,XGBoost,0.7888,0.7851,0.8853,0.8463
4,MLP (20 epochs),0.7291,0.8139,0.8626,0.8632


Three-window context increased mean validation transition F1 by about 0.122 and mean test transition F1 by about 0.122 over the direct baseline. Weighting alone added about 0.026 on validation and 0.013 on test on average, with mixed per-model effects. Context is the clearest source of improvement; weighting is a modest, model-dependent adjustment.

## Results: oversampling on top of context and weights

ROS was applied only inside each training fold, after participant grouping. Transition classes were randomly copied with replacement to a target of 60 or 100 rows; validation and test remained naturally imbalanced. The copies inherit the original training-fold class weights.

In [6]:
oversampling = pd.DataFrame({
    "Model": models,
    "CV no ROS": [0.8631, 0.8649, 0.8236, 0.8398, 0.8617],
    "CV ROS 60": [0.8641, 0.8669, 0.8373, 0.8359, 0.8391],
    "CV ROS 100": [0.8668, 0.8655, 0.8231, 0.8192, 0.8346],
    "Test no ROS": [0.8966, 0.8778, 0.8727, 0.8463, 0.8632],
    "Test ROS 60": [0.8966, 0.8778, 0.8819, 0.8587, 0.8824],
})
oversampling

,Model,CV no ROS,CV ROS 60,CV ROS 100,Test no ROS,Test ROS 60
0,Logistic regression,0.8631,0.8641,0.8668,0.8966,0.8966
1,Linear SVM,0.8649,0.8669,0.8655,0.8778,0.8778
2,Random forest,0.8236,0.8373,0.8231,0.8727,0.8819
3,XGBoost,0.8398,0.8359,0.8192,0.8463,0.8587
4,MLP (20 epochs),0.8617,0.8391,0.8346,0.8632,0.8824


In [7]:
rf_ros_confirmation = pd.DataFrame({
    "Fold seed": [42, 123, 2026, "All nine paired folds"],
    "No ROS transition F1": [0.8236, 0.8576, 0.8238, 0.8350],
    "ROS 60 transition F1": [0.8373, 0.8613, 0.8367, 0.8451],
    "Common F1 change": [-0.0026, -0.0041, -0.0020, -0.0029],
})
rf_ros_confirmation

,Fold seed,No ROS transition F1,ROS 60 transition F1,Common F1 change
0,42,0.8236,0.8373,-0.0026
1,123,0.8576,0.8613,-0.0041
2,2026,0.8238,0.8367,-0.0020
3,All nine paired folds,0.8350,0.8451,-0.0029


The random-forest ROS-60 gain was the most repeatable resampling result: +0.0101 mean transition F1 across nine paired folds, with 8/9 folds improving. Its test common-class F1 and accuracy declined slightly. ROS-60 was neutral or harmful for several other validation runs; target 100 was not consistently better. Treat the test split as descriptive.

## Results: model and PCA comparison

PCA reduced the 1,683 contextual features to 200 components, fitted within each training fold. These are fixed-configuration comparisons, not a claim that PCA is universally harmful.

In [8]:
pca_comparison = pd.DataFrame({
    "Model": ["Logistic regression", "Linear SVM", "Random forest", "XGBoost", "MLP (40 epochs)"],
    "Raw transition F1": [0.8631, 0.8649, 0.8236, 0.8398, 0.8409],
    "PCA-200 transition F1": [0.8405, 0.8133, 0.7422, 0.7401, 0.8430],
    "Raw common F1": [0.9298, 0.9383, 0.9027, 0.9195, 0.9216],
    "PCA-200 common F1": [0.9198, 0.9175, 0.8621, 0.8874, 0.9215],
})
mlp_epochs = pd.DataFrame({"Epochs": [20, 40, 80], "Raw transition F1": [0.8617, 0.8409, 0.8195], "PCA-200 transition F1": [0.8503, 0.8430, 0.8540]})
pca_comparison, mlp_epochs

(                 Model  Raw transition F1  ...  Raw common F1  PCA-200 common F1
0  Logistic regression             0.8631  ...         0.9298             0.9198
1           Linear SVM             0.8649  ...         0.9383             0.9175
2        Random forest             0.8236  ...         0.9027             0.8621
3              XGBoost             0.8398  ...         0.9195             0.8874
4      MLP (40 epochs)             0.8409  ...         0.9216             0.9215

[5 rows x 5 columns],    Epochs  Raw transition F1  PCA-200 transition F1
0      20             0.8617                 0.8503
1      40             0.8409                 0.8430
2      80             0.8195                 0.8540)

PCA reduced transition macro-F1 for logistic regression, linear SVM, random forest, and XGBoost in these folds. MLP performance depended on training duration; the 20-epoch raw model was retained for the main matched comparison. Across three validation seeds, the combined logistic model averaged 0.8589 transition F1, compared with 0.8460 for linear SVM and 0.8449 for the 20-epoch MLP.

## Final model comparison: transition metrics first

The following combined-model results use three-window context and mild weighting without oversampling. Test common-class F1 and accuracy are kept separate as secondary context.

In [9]:
final_models = pd.DataFrame({
    "Model": models,
    "Validation transition F1": [0.8631, 0.8649, 0.8236, 0.8398, 0.8617],
    "Test transition F1": [0.8966, 0.8778, 0.8727, 0.8463, 0.8632],
    "Test transition recall": [0.8934, 0.8773, 0.8664, 0.8520, 0.8707],
})
overall_context = pd.DataFrame({
    "Model": models,
    "Test common-class macro-F1": [0.9575, 0.9697, 0.9275, 0.9599, 0.9499],
    "Test accuracy": [0.9529, 0.9630, 0.9250, 0.9541, 0.9437],
})
final_models, overall_context

(                 Model  ...  Test transition recall
0  Logistic regression  ...                  0.8934
1           Linear SVM  ...                  0.8773
2        Random forest  ...                  0.8664
3              XGBoost  ...                  0.8520
4      MLP (20 epochs)  ...                  0.8707

[5 rows x 4 columns],                  Model  Test common-class macro-F1  Test accuracy
0  Logistic regression                      0.9575         0.9529
1           Linear SVM                      0.9697         0.9630
2        Random forest                      0.9275         0.9250
3              XGBoost                      0.9599         0.9541
4      MLP (20 epochs)                      0.9499         0.9437)

In [10]:
transition_class_results = pd.DataFrame({
    "Transition": ["Stand to sit", "Sit to stand", "Sit to lie", "Lie to sit", "Stand to lie", "Lie to stand"],
    "Test support": [23, 10, 32, 25, 49, 27],
    "Baseline F1": [0.8000, 1.0000, 0.6349, 0.7451, 0.6863, 0.6531],
    "Three windows + weights F1": [0.9778, 0.9524, 0.8125, 0.9057, 0.8333, 0.8980],
})
transition_class_results["Change"] = transition_class_results["Three windows + weights F1"] - transition_class_results["Baseline F1"]
transition_class_results

,Transition,Test support,Baseline F1,Three windows + weights F1,Change
0,Stand to sit,23,0.8000,0.9778,0.1778
1,Sit to stand,10,1.0000,0.9524,-0.0476
2,Sit to lie,32,0.6349,0.8125,0.1776
3,Lie to sit,25,0.7451,0.9057,0.1606
4,Stand to lie,49,0.6863,0.8333,0.1470
5,Lie to stand,27,0.6531,0.8980,0.2449


Logistic regression had the highest test transition macro-F1 among the combined models (0.897); linear SVM had the highest test common-class F1 and accuracy, which are secondary here. Five of six logistic-regression transition-class F1 scores increased; sit-to-stand decreased from 1.000 to 0.952 on only ten test windows. The largest remaining pairwise confusion was sit-to-lie versus stand-to-lie.

## Additional exploratory checks and limitations

- Single-window logistic regression sweeps varied C over 0.1, 1, and 10 and class-weight exponent over 0, 0.5, and 1. The best initial grouped-CV transition F1 was 0.7485, only 0.0052 above the best unweighted setting.
- The earlier single-window random-oversampling sweep (target 100) reached 0.7341 CV transition F1. The RBF-SVM sweep reached 0.7179. These were superseded by the matched five-model context comparison.
- A participant-paired bootstrap of the one-window versus selected three-window logistic regression test predictions used 5,000 resamples and seed 42. The test transition-F1 difference was +0.1434, with a percentile interval of [+0.0707, +0.2079]. This interval reflects the nine test participants, not retraining or deployment to a new population.
- Test participants were seen earlier in this project. Do not treat these values as untouched external evaluation. The data are scripted, overlapping HAPT windows; they do not establish free-living transition-event performance.
- The MLP is a reimplementation based on the architecture described in Tianfeng’s report; the original code was unavailable, so this is not a reproduction of its focal-loss model.

To rerun one configuration after installing its optional dependencies, call `evaluate_cv(...)` or `evaluate_test(...)` above. XGBoost requires the `xgboost` package; the MLP requires PyTorch. Full reruns are more expensive than rendering the recorded result tables.